# Restore X 12.44 patched IPA in Google Drive

Seven parts of the exact complete IPA are stored in the connected Google Drive under `My Drive/Xexperiment/parts`. **Run the code cell below** with that same Google account. It combines them within Drive, checks SHA-256 and ZIP integrity, and saves the finished IPA to `My Drive/Xexperiment`.

No ChatGPT download is necessary. The result remains unsigned and native sign-in has not been tested on an iPhone.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib, shutil, zipfile

folder = Path("/content/drive/MyDrive/Xexperiment")
parts = sorted((folder / "parts").glob("X12_44_Patched.ipa.part[0-9][0-9][0-9]"))
assert len(parts) == 7, f"Found {len(parts)}/7 pieces. Use the same Google account as your connected Drive."
dest = folder / "X_12.44_AdFiltered_WebLogin_Full.ipa"
tmp = Path("/content/X_12.44_AdFiltered_WebLogin_Full.ipa")
expected_sha = "48ee819f683bd9f52f40464822e12690c6c86121dac250fad7d9ab480436d523"
expected_size = 304332783

def checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(4*1024*1024), b""):
            h.update(part)
    return h.hexdigest()

if dest.exists() and dest.stat().st_size == expected_size and checksum(dest) == expected_sha:
    print("Already complete:", dest)
else:
    if dest.exists():
        raise ValueError("A partial or incorrect output file already exists. Remove it from Drive before retrying.")
    with tmp.open("wb") as output:
        for p in parts:
            print("Combining", p.name)
            with p.open("rb") as src:
                shutil.copyfileobj(src, output, length=4*1024*1024)
    assert tmp.stat().st_size == expected_size, "Combined size mismatch"
    assert checksum(tmp) == expected_sha, "SHA-256 mismatch, do not install"
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None, "IPA ZIP validation failed"
        assert "Payload/Twitter.app/Info.plist" in z.namelist(), "Missing X app bundle"
    shutil.copyfile(tmp, dest)
    assert dest.stat().st_size == expected_size, "Drive upload is incomplete"
    print("COMPLETE. Your IPA is in Google Drive -> My Drive -> Xexperiment")
    print(dest)
print("SHA-256:", expected_sha)
print("Import the IPA into FlareStore to re-sign/install it. Native login is experimental.")
